In [ ]:
# =========================================
# CNN Architectures for Image Classification
# Accuracy and Precision Evaluation
# =========================================

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import precision_score
import numpy as np

# Load and Preprocess Data
(train_images, train_labels), (test_images, test_labels) = (
    keras.datasets.mnist.load_data()
)

# Reshape for CNN Input
train_images = train_images.reshape(
    (train_images.shape[0], 28, 28, 1)
)

test_images = test_images.reshape(
    (test_images.shape[0], 28, 28, 1)
)

# Normalize
train_images = train_images / 255.0
test_images = test_images / 255.0

# Model 1: Simple CNN
def build_simple_cnn():
    model = keras.Sequential([
        layers.Conv2D(
            32,
            (3, 3),
            activation='relu',
            input_shape=(28, 28, 1)
        ),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(
            64,
            (3, 3),
            activation='relu'
        ),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    return model

# Model 2: Deep CNN
def build_deep_cnn():
    model = keras.Sequential([
        layers.Conv2D(
            32,
            (3, 3),
            activation='relu',
            padding='same',
            input_shape=(28, 28, 1)
        ),
        layers.Conv2D(
            32,
            (3, 3),
            activation='relu'
        ),
        layers.MaxPooling2D(),
        layers.Dropout(0.25),

        layers.Conv2D(
            64,
            (3, 3),
            activation='relu',
            padding='same'
        ),
        layers.Conv2D(
            64,
            (3, 3),
            activation='relu'
        ),
        layers.MaxPooling2D(),
        layers.Dropout(0.25),

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax')
    ])
    return model

# Model 3: Transfer Learning Model
def build_transfer_model():

    input_tensor = keras.Input(shape=(28, 28, 1))

    x = layers.Lambda(
        lambda image: tf.image.resize(image, (32, 32))
    )(input_tensor)

    x = layers.Lambda(
        lambda image: tf.image.grayscale_to_rgb(image)
    )(x)

    base_model = keras.applications.MobileNetV2(
        input_shape=(32, 32, 3),
        include_top=False,
        weights=None
    )

    x = base_model(x)

    x = layers.GlobalAveragePooling2D()(x)

    output_tensor = layers.Dense(
        10,
        activation='softmax'
    )(x)

    model = keras.Model(
        inputs=input_tensor,
        outputs=output_tensor
    )

    return model

# Compile and Train
def compile_and_train(model):

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    history = model.fit(
        train_images,
        train_labels,
        epochs=10,
        validation_split=0.2,
        batch_size=64
    )

    return history

# Evaluate Performance
def evaluate_model(model):

    test_loss, test_acc = model.evaluate(
        test_images,
        test_labels,
        verbose=0
    )

    y_pred = model.predict(test_images)

    y_pred_classes = np.argmax(
        y_pred,
        axis=1
    )

    precision = precision_score(
        test_labels,
        y_pred_classes,
        average='macro'
    )

    return test_acc, precision

# Run Experiments
models = {
    "Simple CNN": build_simple_cnn(),
    "Deep CNN": build_deep_cnn(),
    "Transfer Model": build_transfer_model()
}

results = {}

for name, model in models.items():

    print(f"Training {name}...")

    compile_and_train(model)

    acc, prec = evaluate_model(model)

    results[name] = {
        "Accuracy": acc,
        "Precision": prec
    }

print(results)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training Simple CNN...
Epoch 1/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 44s 54ms/step - accuracy: 0.9415 - loss: 0.1998 - val_accuracy: 0.9685 - val_loss: 0.1005
Epoch 2/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 38s 51ms/step - accuracy: 0.9818 - loss: 0.0584 - val_accuracy: 0.9851 - val_loss: 0.0487
Epoch 3/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 38s 51ms/step - accuracy: 0.9876 - loss: 0.0402 - val_accuracy: 0.9778 - val_loss: 0.0729
Epoch 4/10
433/750 ━━━━━━━━━━━━━━━━━━━━ 15s 48ms/step - accuracy: 0.9892 - loss: 0.0351